### 獲取財報資訊

這個單元將教您如何獲取財報
首先我們要先看一下財報的網址：
https://mopsov.twse.com.tw/server-java/t164sb01?step=1&CO_ID=1101&SYEAR=2017&SSEASON=3&REPORT_ID=C
在這一串網址中，有幾個重要的元素：

- step: 1 （不知道做啥用的）
- CO_ID: 1101 （股票代號）
- SYEAR: 2017 （年）
- SSEASON: 3 （季）
- REPORT_ID：個別財報(A) 個體財報(B) 合併報表(C)

這邊的report ID，我們必須要以合併報表優先（90%以上的財報都是合併財報），假如沒有合併報表，我們在抓個體財報喔！


In [3]:
# 將網站給爬下來
import requests
import os
from dotenv import load_dotenv

# 載入 .env 文件
load_dotenv()

# 從環境變數讀取代理設置，如果為空則設為 None
http_proxy = os.getenv('HTTP_PROXY') or None
https_proxy = os.getenv('HTTPS_PROXY') or None

# 只有當代理不為空時才設置
proxies = None
if http_proxy or https_proxy:
    proxies = {
        'http': http_proxy,
        'https': https_proxy,
    }


res = requests.get(
    'https://mopsov.twse.com.tw/server-java/t164sb01?step=1&CO_ID=1101&SYEAR=2017&SSEASON=3&REPORT_ID=C',
    proxies=proxies,
    timeout=10,
)
res.encoding = 'big5'

In [4]:
from io import StringIO
import pandas as pd

# 將 res.text 用 StringIO 轉成 檔案 再用 pd.read_html 將 html文字檔轉成 dataframe
dfs = pd.read_html(StringIO(res.text))

In [5]:
dfs[1]

,會計項目,2017年09月30日,2016年12月31日,2016年09月30日
,資產負債表,資產負債表,資產負債表,資產負債表
0,資產,NaN,NaN,NaN
1,流動資產,NaN,NaN,NaN
2,現金及約當現金,NaN,NaN,NaN
3,現金及約當現金總額,26608300.0,28179758.0,27229371.0
4,透過損益按公允價值衡量之金融資產－流動,NaN,NaN,NaN
...,...,...,...,...
96,非控制權益,40744347.0,40628620.0,40281876.0
97,權益總額,152966569.0,147396671.0,145457904.0
98,負債及權益總計,269457896.0,266988696.0,265077946.0


In [10]:
# for sid in ['1101', '2330']:
import time
import os

# 假如沒有 course9 這個資料夾
# if 'course9' not in os.listdir():

#     # 就創建一個
#     os.mkdir('course9')

# 想要爬的股票代號
sid = ['1101', '2330']

# 對於每一筆股票代號
for s in sid:

    # 爬取它的html檔
    res = requests.get(
        'https://mopsov.twse.com.tw/server-java/t164sb01?step=1&CO_ID='
        + s
        + '&SYEAR=2017&SSEASON=3&REPORT_ID=C'
    )
    res.encoding = 'big5'

    # 設定存檔的路徑 ex: course9\1101.html
    # path = os.path.join('course9', s + '.html')
    path = os.path.join(s + '.html')

    # 將檔案打開，寫入html，然後關閉
    f = open(path, 'w', encoding='utf-8')
    f.write(res.text)
    f.close()

    print(s)

    # 休息20秒，在跑下一支股票
    time.sleep(20)

1101
2330


In [11]:
dfs = []

# 對於每一支股票
for s in sid:

    # 將檔案 ex: course9\1101.html 拿出來
    # path = os.path.join('course9', s + '.html')
    path = os.path.join(s + '.html')

    # 存在 dfs 中
    dfs.append(pd.read_html(path, encoding='utf-8'))

In [9]:
# 抓取 2330 的第一張dataFrame前10個row

dfs[1][1].head(10)

,0,1,2,3
0,會計項目,2017年09月30日,2016年12月31日,2016年09月30日
1,資產負債表,NaN,NaN,NaN
2,資產,NaN,NaN,NaN
3,流動資產,NaN,NaN,NaN
4,現金及約當現金,NaN,NaN,NaN
5,現金及約當現金合計,408077695,541253833,463971657
6,透過損益按公允價值衡量之金融資產－流動,NaN,NaN,NaN
7,透過損益按公允價值衡量之金融資產－流動合計,1125668,6451112,1848317
8,備供出售金融資產－流動,NaN,NaN,NaN
9,備供出售金融資產－流動淨額,84953011,67788767,45815003
